# SHRED on ME4OH full-field OFAM3 SST data

Experiment 1: 5 randomly located sensors

Experiment 2: 1059 randomly located sensors
- average number of sensors per week in ME4OH SST dataset is 1059 (increases over time)

Experiment 2.5: changing number of sensors and comparing metrics

.

.

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
from collections import Counter

# Third-party imports
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
import numpy as np
import pandas as pd
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from scipy.interpolate import interpn
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.load_sst_en411_data import crop_df_to_area
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import (
    plot_compare_sst_recon,
    plot_sensors,
    plot_sst_map,
)
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette

## Set up

In [ ]:
# Experiment 1: 5 randomly located sensors
# num_sensors = 5
# lags = 52
# exp_index = 1

# Experiment 2: average number of sensors per week in ME4OH SST dataset is 1059 (increases over time)
num_sensors = 1059
lags = 52
exp_index = 2

In [ ]:
# Load data
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations, sensor_coords, test_lag_dates = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
# Load reconstructions
test_recons = np.load(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}.npy")
test_ground_truth = np.load(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}.npy")
test_recons_5 = np.load(f"Reconstructions/Static/Exp1/recons_n5_l52.npy")  # Experiment 1

## Evaluation

In [ ]:
for i in np.random.choice(range(len(test_recons)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_lag_dates[i]}.csv")
    print(f"Date: {test_lag_dates[i]}")
    plot_compare_sst_recon(recon_df, true_df, area="NA")
    plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=sensor_coords)

In [ ]:
# plot_sst_map(recon_df, test_lag_dates[i], min_sst=min(recon_df.SST), max_sst=max(recon_df.SST), area='NA', sensor_coords=sensor_coords)

In [ ]:
# plot_sst_map(true_df, date, min(true_df.SST), max(true_df.SST), area='NA')

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
rmse = np.sqrt(MSE(y_pred=test_recons, y_true=test_ground_truth))
mae = MAE(y_pred=test_recons, y_true=test_ground_truth)
r2 = r2_score(y_pred=test_recons, y_true=test_ground_truth)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(test_ground_truth[i], test_recons[i], 
                   data_range=(max(test_recons[i].max(), test_ground_truth[i].max()) - min(test_recons[i].min(), test_ground_truth[i].min())))
                   for i in range(len(test_recons))])
print(f"Average SSI:\t{ssi:.3f}")

#### Gulf-Stream region

In [ ]:
GS_test_data = mask_array_by_lat_long(test_recons, lats, longs)
GS_truth_data = mask_array_by_lat_long(test_ground_truth, lats, longs)
GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

In [ ]:
rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i], data_range=test_recons[i].max() -  test_recons[i].min()) for i in range(len(test_recons))])
print(f"Average SSI:\t{ssi:.3f}")

### Density scatter plot  (NEEDS UPDATE)

In [ ]:
def density_scatter( x , y, ax = None, sort = True, bins = 20, **kwargs )   :
    """Scatter plot colored by 2d histogram"""
    if ax is None :
        fig , ax = plt.subplots()
    data , x_e, y_e = np.histogram2d( x, y, bins = bins, density = True )
    z = interpn( ( 0.5*(x_e[1:] + x_e[:-1]) , 0.5*(y_e[1:]+y_e[:-1]) ) , data , np.vstack([x,y]).T , method = "splinef2d", bounds_error = False)

    #To be sure to plot all data
    z[np.where(np.isnan(z))] = 0.0

    # Sort the points by density, so that the densest points are plotted last
    if sort :
        idx = z.argsort()
        x, y, z = x[idx], y[idx], z[idx]

    plt.scatter( x, y, c=z, **kwargs )

    norm = Normalize(vmin = np.min(z), vmax = np.max(z))
    cbar = plt.colorbar(cm.ScalarMappable(norm = norm), ax=ax)
    cbar.ax.set_ylabel('Density')

    return ax

In [ ]:
x = test_ground_truth.reshape(-1)
y = test_recons.reshape(-1)
density_scatter( x, y, bins=[30,30], s=2)

In [ ]:
# https://stackoverflow.com/questions/20105364/how-can-i-make-a-scatter-plot-colored-by-density

# "Viridis-like" colormap with white background
white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)

def using_mpl_scatter_density(fig, x, y):
    """Use mpl scatter density"""
    ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
    density = ax.scatter_density(x, y, cmap=white_viridis)
    ax.set_xlim(-5, 35)
    ax.set_ylim(-5, 35)
    fig.colorbar(density, label='Number of points per pixel')

fig = plt.figure()
x = test_ground_truth.reshape(-1)
y = test_recons.reshape(-1)
using_mpl_scatter_density(fig, x, y)
plt.plot([-5, 35], [-5, 35], ':', lw=1, c='r')
plt.xlabel("Observed SST ($\\degree$C)")
plt.ylabel("Reconstructed SST ($\\degree$C)")
plt.show()

In [ ]:
fig = plt.figure()
i = 177
# plt.scatter(test_ground_truth[i], test_recons[i], alpha=0.25)
using_mpl_scatter_density(fig, test_ground_truth[i].reshape(-1), test_recons[i].reshape(-1))
plt.plot([-5, 35], [-5, 35], ':', lw=1, c='r')
plt.xlabel("Observed SST ($\\degree$C)")
plt.ylabel("Reconstructed SST ($\\degree$C)")
plt.title(f"{test_lag_dates[i]}")

In [ ]:
# lats = np.array([20.0, 31.0, 33.0, 50.0, 55.0])
# longs = np.array([-80.0, -80.0, -34.0, -34.0, -34.0])
# data = np.array([[24.0, 25.0, 26.0, 27.0, 28.0], [24.0, 25.0, 26.0, 27.0, 28.0], [24.0, 25.0, 26.0, 27.0, 28.0]])
# data = mask_array_by_lat_long(test_recons, lats, longs)

### Regional integral of SST (area weighted mean)

In [ ]:
test_sst_mean, area_test = area_weighted_mean(test_recons, lats, longs)
truth_sst_mean, area_truth = area_weighted_mean(test_ground_truth, lats, longs)
test_sst_mean_5, _ = area_weighted_mean(test_recons_5, lats, longs)

In [ ]:
sst_mean_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED - 5 sensors": test_sst_mean_5,
    "SHRED - 1059 sensors": test_sst_mean,
    "Truth": truth_sst_mean
})
# Smooth over 1 year = 52 weeks
sst_rolling_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED - 5 sensors": sst_mean_df["SHRED - 5 sensors"].rolling(52).mean(),
    "SHRED - 1059 sensors": sst_mean_df["SHRED - 1059 sensors"].rolling(52).mean(),
    "Truth": sst_mean_df.Truth.rolling(52).mean()
})
sst_mean_df = sst_mean_df.set_index("Date")
sst_rolling_df = sst_rolling_df.set_index("Date")

In [ ]:
plt.figure(figsize=(8,8))
sst_mean_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
sst_rolling_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.show()

### Zonally-integrated linear trend in SST
Is this a sensible thing to do?? Need to understand if we have linear data

NOPE - not a linear set of data (see below graphs)

In [ ]:
latitude_avg_sst = []  # Average SST across each latitude band in the data (1/4 degree intervals)
latitude_avg_sst_5 = []
for i in range(len(test_recons)):
    latitude_avg_sst.append(np.array(pd.Series(test_recons[i]).groupby(lats).mean().to_numpy()))
    latitude_avg_sst_5.append(np.array(pd.Series(test_recons_5[i]).groupby(lats).mean().to_numpy()))
latitude_avg_sst = np.array(latitude_avg_sst)

In [ ]:
ten_colours = list(mcolors.TABLEAU_COLORS.values())
# colours_240 = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
#              for i in range(240)]

In [ ]:
# Plot the first ten of these to get an idea of the data shape
# Use transform (.T) to get timeseries for each sensor
for i in range(10):  # latitude_avg_sst.shape[1]):
    plt.plot(latitude_avg_sst.T[i], c=ten_colours[i], label=np.unique(lats)[i])
plt.legend(bbox_to_anchor=(1,1), title="Latitude")
plt.xticks(rotation=90)
plt.ylabel("Zonally averaged SST ($\\degree$C)")
plt.show()

In [ ]:
lat_avg_loc = latitude_avg_sst.T  # get time series for each sensor location

for j in range(10):  # len(lat_avg_loc)):
    sst_lat_mean_df = pd.DataFrame({
        "Date": test_lag_dates,
        "SHRED": lat_avg_loc[j],
    })
    sst_lat_rolling = sst_lat_mean_df["SHRED"].rolling(lags).mean()
    
    plt.plot(test_lag_dates, sst_lat_rolling, c=ten_colours[j], label=np.unique(lats)[j])
    
plt.legend(bbox_to_anchor=(1,1), title="Latitude")
plt.xticks(rotation=90)
plt.ylabel("Smoothed zonally averaged SST ($\\degree$C)")
plt.show()

### Difference between test truth and reconstructions 

In [ ]:
difference = test_recons - test_ground_truth
difference_5 = test_recons_5 - test_ground_truth

#### Plots

In [ ]:
plot_index = 3

In [ ]:
for i in [plot_index]:  #range(0, len(test_recons), 10):  # np.random.choice(range(len(test_recons)), 1):
    diff_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": difference[i]
        })
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_lag_dates[i]}.csv")
    # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
    print(f"Date: {test_lag_dates[i]}")
    plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area="NA")  #, sensor_coords=sensor_coords)

    gs_recon_df = crop_df_to_area(recon_df, "GS", "Latitude")
    gs_true_df = crop_df_to_area(true_df, "GS", "Latitude")
    gs_diff_df = crop_df_to_area(diff_df, "GS", "Latitude")
    
    plot_compare_sst_recon(gs_recon_df, gs_true_df, diff_data=gs_diff_df, area='GS')

In [ ]:
plot_sensors(sensor_coords=sensor_coords)

In [ ]:
min_sst = min([min(recon_df.SST), min(true_df.SST)])
max_sst = max([max(recon_df.SST), max(true_df.SST)])
plot_sst_map(recon_df, title="Reconstructed SST ($\\degree$C)", min_sst=min_sst, max_sst=max_sst, area="NA")

In [ ]:
plot_sst_map(true_df, title="True SST ($\\degree$C)", min_sst=min_sst, max_sst=max_sst, area="NA")

In [ ]:
extent=max(np.abs(min(diff_df.SST)), max(diff_df.SST))
plot_sst_map(diff_df, title="Difference ($\\degree$C)", min_sst=-extent, max_sst=extent, area="NA", cmap='bwr')

## Plot timeseries for each sensor for one element of test set + sensor values of SST distributed spatially

This requires loading the full dataset to have the test data to plot from

Expect: 10 secs

In [ ]:
import random

from sklearn.preprocessing import MinMaxScaler

from Data.preprocess_sst_ofam3_shred import (
    create_shred_sequences,
    load_files,
    split_ordered_data,
)

In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
test_transformed = sc.transform(test_data)

# Build sequences
test_dataset = create_shred_sequences(test_transformed, sensor_locations, lags=lags)

In [ ]:
plot_index = 3
plot_data = test_dataset[plot_index]
random_colours = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
              for i in range(num_sensors)]

for i in range(num_sensors):
    # test_data[0] -> test sequences
    # test_data[0].T -> for each sensor, timeseries of SST values across test dataset
    plt.plot(plot_data[0].T[i], c=random_colours[i])
plt.ylabel("Normalised SST")
plt.xlabel("Week index")
plt.grid()
plt.xlim(0, 52)
plt.show()

In [ ]:
sensor_df = pd.DataFrame({
    "Longitude": sensor_coords[0],
    "Latitude": sensor_coords[1],
    "SST": test_data[plot_index][sensor_locations]
})
plot_sensors(sensor_coords=sensor_coords)
plot_sst_map(sensor_df, area="NA", background=True)

In [ ]:
test_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": test_data[plot_index]
})
plot_sst_map(test_df, area="NA", background=True)

In [ ]:
test_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": test_recons[plot_index]
})
plot_sst_map(test_df, area="NA", background=True)

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

OR could scale by area by doing lat_band/total_area????

In [ ]:
latitude_avg_diffs = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference])
avg_all_time_lat_avg_diffs = np.array([np.mean(latitude_avg_diffs.T[i]) for i in range(latitude_avg_diffs.shape[1])])

latitude_avg_diffs_5 = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference_5])
avg_all_time_lat_avg_diffs_5 = np.array([np.mean(latitude_avg_diffs_5.T[i]) for i in range(latitude_avg_diffs_5.shape[1])])

In [ ]:
plt.plot(avg_all_time_lat_avg_diffs_5, np.unique(lats), c=colours[0], label="5 sensors")
plt.plot(avg_all_time_lat_avg_diffs, np.unique(lats), c=colours[1], label="1059 sensors")
plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend()

NOTE: not even numbers of data points in each year

In [ ]:
years = [int(date.year) for date in test_lag_dates]
year_labels = np.unique(years)
Counter(years)

In [ ]:
for title, lat_diff in {"5 sensors": latitude_avg_diffs_5, "1059 sensors": latitude_avg_diffs}.items():
    diff_by_years = np.array([[time[j] for time in lat_diff] for j in range(len(lat_diff[0]))])
    latitude_avg_diffs_avg_by_year = np.array([pd.Series(d).groupby(years).mean().to_numpy() for d in diff_by_years])

    for i in range(len(year_labels)):
        lat_for_year = [lat[i] for lat in latitude_avg_diffs_avg_by_year]
        plt.plot(lat_for_year, np.unique(lats), label=year_labels[i], c=colours[i])
    plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
    plt.xlabel('Average difference (from the truth) ($\\degree$C)')
    plt.ylabel('Latitude')
    plt.title(title)
    plt.legend()
    plt.show()

#### Average difference between reconstruction and truth across whole NA region - by year

In [ ]:
avg_difference = [np.mean(d) for d in difference]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

In [ ]:
# For Gulf Stream only:
GS_data = mask_array_by_lat_long(difference, lats, longs)
GS_data = [d[~np.isnan(d)] for d in GS_data]

In [ ]:
avg_difference = [np.mean(d) for d in GS_data]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

### Differences between test reconstructions
Verifying that not just getting identical reconstructions
- Not every point in scatter plot is 0 - so there is difference!

Also see: clear seasonality in the generations being produced!
- Sensible that there's a clear difference when spring to autumn are compared

In [ ]:
# avg_diff = []
# date1 = []
# date2 = []
# for i in range(len(test_recons)):
#     for j in range(len(test_recons)):
#         date1.append(test_dates[i])
#         date2.append(test_dates[j])
#         avg_diff.append(np.mean(test_recons[i] - test_recons[j]))

# diff_df = pd.DataFrame({
#     "Date 1": date1,
#     "Date 2": date2,
#     "Average difference ($\\degree$C)": avg_diff
# })

In [ ]:
# s = diff_df.plot.scatter(x="Date 1", y="Date 2", c="Average difference ($\\degree$C)", cmap='rainbow')
# plt.xticks(rotation=90)
# plt.grid()

In [ ]:
# Santity check that all reconstructions aren't identical
bool_check = [[np.array_equal(recon1,recon2) for recon2 in test_recons] for recon1 in test_recons]

In [ ]:
# Check that only ever 1 True value (on diagonal)
Counter([Counter(row) == Counter({False: 182, True: 1}) for row in bool_check])
# Expected output: Counter({True: 183})

In [ ]:
# Check all diagonals are true
Counter(np.diag(bool_check))
# Expected output: Counter({True: 183})

In [ ]:
plt.matshow(bool_check)
# Expected output: Only diagonal of matrix is True (yellow), rest is False (purple)